# Siren vs. Non-Siren Sound Classification with a Small CNN

**Mini-project problem statement:** Build a beginner-friendly Convolutional Neural Network (CNN) that classifies short urban audio clips into two classes: **siren** and **non-siren**. Audio is converted into a mel-spectrogram, a visual representation of sound energy across time and frequency, which the CNN uses as input.

**Dataset:** UrbanSound8K (8,732 labeled clips, up to 4 seconds, 10 classes). The dataset's official labels include `siren` with `classID = 8`. All other nine categories will be grouped as `non-siren`.

**Important:** UrbanSound8K is provided under dataset-specific terms; review the official terms before using it. This repository contains code only, not the dataset.

References: [Soundata repository](https://github.com/soundata/soundata), [UrbanSound8K official page](https://urbansounddataset.weebly.com/urbansound8k.html), Salamon, Jacoby & Bello (2014), *A Dataset and Taxonomy for Urban Sound Research*.

## 1. Install and import libraries

Run this notebook in Google Colab. The first download of UrbanSound8K can take a while and needs sufficient disk space.

In [ ]:
!pip -q install soundata librosa scikit-learn matplotlib pandas tensorflow

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundata

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import tensorflow as tf
from tensorflow.keras import layers, models

np.random.seed(42)
tf.random.set_seed(42)

## 2. Download and validate UrbanSound8K

Soundata provides a consistent way to download the dataset, validate files, and load audio clips. This follows the repository's documented workflow.

In [ ]:
dataset = soundata.initialize('urbansound8k')
dataset.download()
missing_files, invalid_files = dataset.validate()
print('Dataset location:', dataset.data_home)
print('Missing files:', len(missing_files))
print('Files with checksum issues:', len(invalid_files))

## 3. Inspect the metadata

The CSV includes the filename, fold, class ID, and class label. The original dataset is arranged into ten folds; here, fold 10 is held out for final testing and folds 1–9 are used for model development.

In [ ]:
metadata_path = os.path.join(dataset.data_home, 'metadata', 'UrbanSound8K.csv')
metadata = pd.read_csv(metadata_path)

print('Total metadata rows:', len(metadata))
print('Available columns:', metadata.columns.tolist())
print('Counts per original sound class:')
print(metadata['class'].value_counts().sort_index())

metadata.head()

In [ ]:
# Keep siren as class 1 and all other sounds as class 0.
metadata['target'] = (metadata['class'] == 'siren').astype(int)
metadata['clip_id'] = metadata['slice_file_name'].str.replace('.wav', '', regex=False)

print('Binary class counts:')
print(metadata['target'].map({0: 'non-siren', 1: 'siren'}).value_counts())

metadata['fold'].value_counts().sort_index().plot(kind='bar', title='Number of clips in each fold')
plt.xlabel('Fold')
plt.ylabel('Number of clips')
plt.show()

## 4. Make a small, balanced dataset

To keep this mini-project practical on a free Colab runtime, we use up to 500 examples per class for model development and balance the final test set using examples from fold 10. The model is trained on a subset rather than every clip. Increase `TRAIN_PER_CLASS` for a larger experiment.

In [ ]:
TRAIN_PER_CLASS = 500

train_pool = metadata[metadata['fold'] != 10]
test_pool = metadata[metadata['fold'] == 10]

train_parts = []
for label in [0, 1]:
    part = train_pool[train_pool['target'] == label]
    train_parts.append(part.sample(n=min(TRAIN_PER_CLASS, len(part)), random_state=42))
train_df = pd.concat(train_parts).sample(frac=1, random_state=42).reset_index(drop=True)

# Balance the test classes using clips from fold 10.
test_counts = test_pool['target'].value_counts()
test_n = int(min(test_counts.get(0, 0), test_counts.get(1, 0)))
test_df = pd.concat([
    test_pool[test_pool['target'] == 0].sample(n=test_n, random_state=42),
    test_pool[test_pool['target'] == 1].sample(n=test_n, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

# Split development data into training and validation, preserving class balance.
train_df, val_df = train_test_split(
    train_df, test_size=0.2, random_state=42, stratify=train_df['target']
)
print('Train:', len(train_df), '| Validation:', len(val_df), '| Test:', len(test_df))
print('Train class counts:', train_df['target'].value_counts().to_dict())
print('Validation class counts:', val_df['target'].value_counts().to_dict())
print('Test class counts:', test_df['target'].value_counts().to_dict())

## 5. Convert audio into mel-spectrograms

A spectrogram is an image-like view of sound. The horizontal direction represents time, the vertical direction represents frequency, and brighter values mean more energy. Every clip is converted to the same input shape so the CNN can process a batch.

In [ ]:
SAMPLE_RATE = 22050
AUDIO_SECONDS = 4
SAMPLES = SAMPLE_RATE * AUDIO_SECONDS
N_MELS = 64
MAX_FRAMES = 173

def make_spectrogram(clip_id):
    # Soundata's clip loader returns (mono_audio, sample_rate).
    audio, sr = dataset.clip(clip_id).audio
    if sr != SAMPLE_RATE:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=SAMPLE_RATE)

    # Pad short clips or trim longer clips to four seconds.
    audio = librosa.util.fix_length(audio, size=SAMPLES)
    mel = librosa.feature.melspectrogram(
        y=audio, sr=SAMPLE_RATE, n_fft=1024, hop_length=512, n_mels=N_MELS
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = librosa.util.fix_length(mel_db, size=MAX_FRAMES, axis=1)
    return mel_db

# Preview an example before processing the subset.
example_id = train_df.iloc[0]['clip_id']
example_mel = make_spectrogram(example_id)
plt.figure(figsize=(8, 3))
librosa.display.specshow(example_mel, sr=SAMPLE_RATE, hop_length=512, x_axis='time', y_axis='mel')
plt.title('Example mel-spectrogram')
plt.colorbar(format='%+2.0f dB')
plt.tight_layout()
plt.show()

## 6. Prepare training, validation, and test arrays

This is the most time-consuming preprocessing section because each selected audio file must be read and converted. Computed spectrograms are kept in memory for this run.

In [ ]:
def build_arrays(frame):
    features = []
    labels = []
    for _, row in frame.iterrows():
        try:
            features.append(make_spectrogram(row['clip_id']))
            labels.append(int(row['target']))
        except Exception as error:
            print('Skipping unreadable clip:', row['slice_file_name'], error)
    X = np.array(features, dtype=np.float32)
    y = np.array(labels, dtype=np.float32)
    return X, y

X_train, y_train = build_arrays(train_df)
X_val, y_val = build_arrays(val_df)
X_test, y_test = build_arrays(test_df)

# Scale decibel values to a stable range and add CNN's channel dimension.
X_train = ((X_train + 80) / 80).clip(0, 1)[..., np.newaxis]
X_val = ((X_val + 80) / 80).clip(0, 1)[..., np.newaxis]
X_test = ((X_test + 80) / 80).clip(0, 1)[..., np.newaxis]

print('Input shape:', X_train.shape)
print('Labels: 0 = non-siren, 1 = siren')

## 7. Build a small CNN

The model has two convolutional layers, pooling, a small dense layer, and one sigmoid output. ReLU is used in hidden layers; sigmoid outputs a probability for the siren class. Binary cross-entropy is the loss function for two classes.

In [ ]:
model = models.Sequential([
    layers.Input(shape=(N_MELS, MAX_FRAMES, 1)),
    layers.Conv2D(8, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(16, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## 8. Train the CNN

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=8,
    batch_size=16,
    verbose=1
)

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training')
plt.plot(history.history['val_loss'], label='Validation')
plt.title('Loss')
plt.xlabel('Epoch')
plt.legend()
plt.tight_layout()
plt.show()

## 9. Evaluate on the held-out test fold

The test set is not used to train the model. Accuracy is useful, but also inspect precision and recall because the model might make different kinds of mistakes.

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
probabilities = model.predict(X_test, verbose=0).ravel()
predictions = (probabilities >= 0.5).astype(int)

print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')
print(classification_report(y_test.astype(int), predictions, target_names=['non-siren', 'siren']))

ConfusionMatrixDisplay.from_predictions(
    y_test.astype(int), predictions,
    display_labels=['non-siren', 'siren'], values_format='d'
)
plt.title('Test confusion matrix')
plt.show()

## 10. Try one test clip

This example predicts one audio clip selected from the held-out test subset.

In [ ]:
index = 0
probability = float(model.predict(X_test[index:index+1], verbose=0)[0][0])
predicted_label = 'siren' if probability >= 0.5 else 'non-siren'
actual_label = 'siren' if int(y_test[index]) == 1 else 'non-siren'

print('Actual class:', actual_label)
print('Predicted class:', predicted_label)
print(f'Predicted siren probability: {probability:.3f}')

## Conclusion

This notebook demonstrates a small binary sound-classification pipeline: load metadata and audio, create mel-spectrograms, train a CNN, and evaluate the results. Results depend on the selected subset, random sampling, and runtime. Do not claim a particular accuracy before running the notebook. For a stronger experiment, increase the per-class training sample count, compare validation/test results across runs, and document errors.